# K-Means Clustering using Telecom Customer Dataset

This notebook demonstrates **unsupervised machine learning** using the **K-Means clustering algorithm**.

We will use a telecom customer dataset with 500 records and 10 columns.

Main steps:

1. Load the dataset  
2. Understand the data  
3. Prepare features  
4. Apply scaling  
5. Use the Elbow Method to find the best number of clusters  
6. Train K-Means  
7. Analyze customer clusters  
8. Save final clustered output


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score


## 2. Load the Telecom Dataset

In [ ]:
# Load CSV file
df = pd.read_csv("telecom_customer_kmeans_500.csv")

# Display first 5 rows
df.head()


In [ ]:
# Check dataset shape
print("Dataset shape:", df.shape)


In [ ]:
# Display basic information
df.info()


In [ ]:
# Display summary statistics
df.describe()


## 3. Check Missing Values

In [ ]:
# Check missing values in each column
df.isnull().sum()


## 4. Select Features for Clustering

K-Means is an unsupervised algorithm, so we do **not** need a target column.

We remove `customer_id` because it is only an identifier.

The column `contract_type` is categorical, so we convert it into numeric columns using one-hot encoding.


In [ ]:
# Remove customer_id column
X = df.drop("customer_id", axis=1)

# Convert categorical column into numeric columns
X = pd.get_dummies(X, columns=["contract_type"], drop_first=True)

# Display final feature columns
X.head()


## 5. Scale the Data

K-Means uses distance calculation.

So scaling is very important because columns like `total_charges`, `call_minutes`, and `sms_count` have very different ranges.


In [ ]:
# Create scaler object
scaler = StandardScaler()

# Scale all input features
X_scaled = scaler.fit_transform(X)

print("Scaling completed")


## 6. Elbow Method

The Elbow Method helps us choose the best value of **K**, which means the number of clusters.

WCSS means **Within Cluster Sum of Squares**.

Lower WCSS is better, but after some point the improvement becomes very small.

That bending point is called the **elbow point**.


In [ ]:
# Store WCSS values
wcss = []

# Try cluster values from 1 to 10
for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Display WCSS values
for k, value in zip(range(1, 11), wcss):
    print("K =", k, "WCSS =", value)


In [ ]:
# Plot Elbow Method graph
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker="o")
plt.xlabel("Number of Clusters K")
plt.ylabel("WCSS / Inertia")
plt.title("Elbow Method for Finding Best K")
plt.grid(True)
plt.show()


## 7. Train Final K-Means Model

From the elbow graph, choose a suitable K value.

For this notebook, we are using **K = 3**.

You can change this value based on the elbow graph.


In [ ]:
# Final K-Means model
final_k = 3

kmeans = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=10
)

# Fit model and create cluster labels
df["cluster"] = kmeans.fit_predict(X_scaled)

# Display first 10 records
df.head(10)


## 8. Cluster Count

This shows how many customers are present in each cluster.


In [ ]:
df["cluster"].value_counts()


## 9. Cluster Analysis

This helps us understand the average behavior of customers in each cluster.


In [ ]:
cluster_summary = df.groupby("cluster").mean(numeric_only=True)

cluster_summary


## 10. Validation Metrics for Clustering

For unsupervised learning, we do not normally use accuracy or confusion matrix.

Common clustering validation metrics are:

- **Silhouette Score**: Higher is better
- **Davies-Bouldin Score**: Lower is better
- **Calinski-Harabasz Score**: Higher is better


In [ ]:
sil_score = silhouette_score(X_scaled, df["cluster"])
db_score = davies_bouldin_score(X_scaled, df["cluster"])
ch_score = calinski_harabasz_score(X_scaled, df["cluster"])

print("Silhouette Score:", sil_score)
print("Davies-Bouldin Score:", db_score)
print("Calinski-Harabasz Score:", ch_score)


## 11. Simple Cluster Interpretation

You can interpret clusters based on the summary table.

Example:

- Cluster 0: Low usage / low billing customers
- Cluster 1: High usage / high billing customers
- Cluster 2: Customers with higher service calls or possible churn risk

Actual interpretation depends on the values shown in `cluster_summary`.


## 12. Save Clustered Output

In [ ]:
# Save final dataset with cluster labels
df.to_csv("telecom_customer_kmeans_clustered_output.csv", index=False)

print("Clustered output file saved successfully")
